In [ ]:
!pip install esig iisignature scikit-learn pandas numpy matplotlib scipy aeon
import numpy as np
import iisignature
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from aeon.datasets import load_classification
from itertools import combinations
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
import pandas as pd
from datetime import datetime
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr


@dataclass
class Config:
    """Configuration with channel selection."""
    sig_level: int = 2
    n_estimators: int = 100
    random_state: int = 42
    max_features: int = 250
    k_folds: int = 5
    start_k: int = 2
    max_k: int = 2
    max_combinations_per_k: int = 10000000000000000000


# ════════════════════════════════════════════════════════════════════════════
# Channel augmentation — many channels for low-degree regime
#
# Strategy: at sig_level=2 and max_k=2, depth is capped so we compensate
# with WIDTH — more diverse channels create more diverse pairwise interactions.
#
# Fairness note: channels that would bias higher-degree signatures are now
# fair game because sig_level=2 cannot access them:
#   x²      → area(x_i, x_j²) = level-3 sig term  → sig at level 2 CANNOT compute it
#   cumsum  → area(x_i, cumsum_j) ≈ level-2 sig term BUT sig already has this,
#             so it benefits our features with an equivalent the sig already sees
#   time    → same argument as cumsum at level 2
#
# No normalised copies — StandardScaler runs on train fold before every RF fit,
# so normalised channels produce only linearly scaled areas (redundant).
#
# 8 channel blocks (8× original channels):
#   x         — raw signal
#   curvature — local bending radius (nonlinear, not low-order sig term)
#   chord     — linear interpolation start→end; area(x_i, chord_j) has no
#               clean sig equivalent at any level
#   |x|       — magnitude envelope; d|x| flips at zero crossings,
#               inexpressible at any finite sig level
#   cumsum    — integrated path; at level 2, sig already has ∫x_i dx_j,
#               cumsum adds the complementary ∫(∫x_j) term
#   x²        — squared signal; area(x_i, x_j²) is a level-3 sig term,
#               so sig at level 2 has NO access to this
#   dx        — increments; at level 2 area(x_i, dx_j)=level-2 sig term,
#               but area(curvature_i, dx_j) and area(|x_i|, dx_j) are new
#   smooth(x) — Gaussian smoothed (σ=T/8); d(smooth) is non-local,
#               inexpressible as polynomial in x and dx
# ════════════════════════════════════════════════════════════════════════════

def _curvature_radius(data: np.ndarray) -> np.ndarray:
    """(n_ch, T) → (n_ch, T) local curvature radius per channel."""
    n_channels, n_timepoints = data.shape
    padded = np.zeros((n_channels, n_timepoints + 2))
    padded[:, 0]    = data[:, 0]
    padded[:, 1:-1] = data
    padded[:, -1]   = data[:, -1]
    first_deriv  = padded[:, 2:] - padded[:, :-2]
    second_deriv = padded[:, 2:] - 2 * padded[:, 1:-1] + padded[:, :-2]
    radius = (1 + first_deriv ** 2) ** 1.5 / (np.abs(second_deriv) + 1e-6)
    return np.clip(radius / 1000, 0.001, 100)


def _gaussian_smooth(data: np.ndarray, sigma_fraction: float = 0.125) -> np.ndarray:
    """(n_ch, T) → (n_ch, T) Gaussian smoothed with σ = sigma_fraction × T."""
    n_ch, T  = data.shape
    sigma    = max(1.0, sigma_fraction * T)
    radius   = int(3 * sigma)
    kt       = np.arange(-radius, radius + 1)
    kernel   = np.exp(-0.5 * (kt / sigma) ** 2)
    kernel  /= kernel.sum()
    out      = np.zeros_like(data)
    for c in range(n_ch):
        out[c] = np.convolve(data[c], kernel, mode='same')
    return out


def _per_sample_normalise(data: np.ndarray) -> np.ndarray:
    """
    (n_ch, T) → (n_ch, T)  per-channel, per-sample normalisation.

    Normalises each channel to zero mean and unit std over time, WITHIN
    a single sample.  This is orthogonal to StandardScaler which normalises
    each feature ACROSS samples:

        per-sample norm  →  removes amplitude differences between channels
                            inside one sample (shape/relative geometry)
        StandardScaler   →  removes amplitude differences between samples
                            (across-sample scale)

    Consequence for areas:
        area(x_i, x_j)            — absolute geometry, amplitude-dependent
        area(x_i_norm, x_j_norm)  — shape geometry, amplitude-invariant

    Two samples with identical shape but different amplitudes have IDENTICAL
    normalised areas but DIFFERENT unnormalised areas — genuinely new info.
    """
    means = np.mean(data, axis=1, keepdims=True)
    stds  = np.std(data,  axis=1, keepdims=True)
    stds  = np.where(stds < 1e-8, 1.0, stds)
    return (data - means) / stds


def augment_channels(x: np.ndarray) -> np.ndarray:
    """
    (n_ch, T) → (8*n_ch, T)

    4 raw channels — absolute geometry:
        0·n_ch : 1·n_ch  →  x
        1·n_ch : 2·n_ch  →  chord
        2·n_ch : 3·n_ch  →  curvature
        3·n_ch : 4·n_ch  →  |x|

    4 per-sample normalised copies of the same 4:
        4·n_ch : 5·n_ch  →  x_norm
        5·n_ch : 6·n_ch  →  chord_norm
        6·n_ch : 7·n_ch  →  curvature_norm
        7·n_ch : 8·n_ch  →  |x|_norm

    Raw  → area captures absolute amplitude interactions
    Norm → area captures shape interactions independent of per-channel scale
    """
    n_ch, T = x.shape

    t     = np.linspace(0, 1, T)
    chord = x[:, [0]] + (x[:, [-1]] - x[:, [0]]) * t

    raw    = [x, chord, _curvature_radius(x), np.abs(x)]
    normed = [_per_sample_normalise(b) for b in raw]

    return np.vstack(raw + normed)                                 # (8*n_ch, T)


# ════════════════════════════════════════════════════════════════════════════
# Feature extraction (unchanged from original)
# ════════════════════════════════════════════════════════════════════════════

def vectorized_pairwise_volumes(
    X: List[np.ndarray],
    combinations_list: List[Tuple[int, int]],
) -> Tuple[np.ndarray, np.ndarray]:
    n_samples      = len(X)
    n_combinations = len(combinations_list)
    signed_features   = np.zeros((n_samples, n_combinations))
    unsigned_features = np.zeros((n_samples, n_combinations))
    for sample_idx, x in enumerate(X):
        for combo_idx, (i, j) in enumerate(combinations_list):
            ch_i = x[i];  ch_j = x[j]
            diff_i = np.diff(ch_i);  diff_j = np.diff(ch_j)
            sum_i  = ch_i[1:] + ch_i[:-1]
            sum_j  = ch_j[1:] + ch_j[:-1]
            areas  = 0.5 * (diff_i * sum_j - diff_j * sum_i)
            areas  = np.clip(areas, -1e10, 1e10)
            signed_features[sample_idx,   combo_idx] = np.clip(np.sum(areas),        -1e10, 1e10)
            unsigned_features[sample_idx, combo_idx] = np.clip(np.sum(np.abs(areas)),-1e10, 1e10)
    return signed_features, unsigned_features


def vectorized_determinant_features(
    X: List[np.ndarray],
    combinations_list: List[Tuple],
    k: int,
) -> Tuple[np.ndarray, np.ndarray]:
    n_samples      = len(X)
    n_combinations = len(combinations_list)
    n_timepoints   = X[0].shape[1]
    signed_features   = np.zeros((n_samples, n_combinations))
    unsigned_features = np.zeros((n_samples, n_combinations))
    for sample_idx, x in enumerate(X):
        for combo_idx, combo in enumerate(combinations_list):
            subset = x[list(combo), :]
            s = u = 0.0
            for t in range(n_timepoints):
                product = 1.0
                for i in range(k):
                    product *= subset[i, t]
                s += product
                u += abs(product)
            signed_features[sample_idx,   combo_idx] = s
            unsigned_features[sample_idx, combo_idx] = u
    return signed_features, unsigned_features


def batch_extract_features(
    X: List[np.ndarray], config: Config
) -> Tuple[np.ndarray, np.ndarray]:
    X_augmented = [augment_channels(x) for x in X]
    n_ch        = X_augmented[0].shape[0]
    print(f"  Augmented to {n_ch} channels ({n_ch // X[0].shape[0]}× = 4 raw + 4 norm: x,chord,curv,|x|)")

    all_combinations = {}
    for k in range(config.start_k, min(config.max_k + 1, n_ch + 1)):
        all_combos = list(combinations(range(n_ch), k))
        if len(all_combos) > config.max_combinations_per_k:
            np.random.seed(42)
            indices    = np.random.choice(len(all_combos),
                                          size=config.max_combinations_per_k,
                                          replace=False)
            all_combos = [all_combos[i] for i in sorted(indices)]
        all_combinations[k] = all_combos
        print(f"  k={k}: {len(all_combos)} combinations "
              f"[est. {2 * len(all_combos) * len(X) * 8 / 1e9:.3f} GB]")

    all_signed:   List[np.ndarray] = []
    all_unsigned: List[np.ndarray] = []
    for k in range(config.start_k, min(config.max_k + 1, n_ch + 1)):
        combos = all_combinations[k]
        if k == 2:
            sf, uf = vectorized_pairwise_volumes(X_augmented, combos)
        else:
            sf, uf = vectorized_determinant_features(X_augmented, combos, k)
        all_signed.append(sf)
        all_unsigned.append(uf)

    signed_features   = (np.concatenate(all_signed,   axis=1) if all_signed
                         else np.zeros((len(X), 0)))
    unsigned_features = (np.concatenate(all_unsigned, axis=1) if all_unsigned
                         else np.zeros((len(X), 0)))

    signed_features   = np.nan_to_num(signed_features,   nan=0., posinf=1e6, neginf=-1e6)
    unsigned_features = np.nan_to_num(unsigned_features, nan=0., posinf=1e6, neginf=-1e6)
    print(f"  Extracted {signed_features.shape[1]} signed + "
          f"{unsigned_features.shape[1]} unsigned features")
    return signed_features, unsigned_features


# ════════════════════════════════════════════════════════════════════════════
# CV evaluation and experiment runner (unchanged from original)
# ════════════════════════════════════════════════════════════════════════════

def evaluate_with_cv(
    X: np.ndarray, y: np.ndarray, config: Config
) -> Tuple[float, float, float, float, float, float, float, float]:
    if X.shape[1] == 0:
        return 0., 0., 0., 0., 0., 0., 0., 0.

    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    acc_scores_before, f1_scores_before = [], []
    acc_scores_after,  f1_scores_after  = [], []

    for train_idx, val_idx in kfold.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        scaler_before  = StandardScaler()
        X_train_scaled = np.nan_to_num(scaler_before.fit_transform(X_train),
                                       nan=0., posinf=0., neginf=0.)
        X_val_scaled   = np.nan_to_num(scaler_before.transform(X_val),
                                       nan=0., posinf=0., neginf=0.)

        rf_before = RandomForestClassifier(n_estimators=config.n_estimators,
                                           random_state=42, max_depth=10)
        rf_before.fit(X_train_scaled, y_train)
        pred_before = rf_before.predict(X_val_scaled)
        acc_scores_before.append(accuracy_score(y_val, pred_before))
        f1_scores_before.append(f1_score(y_val, pred_before, average='weighted'))

        if X_train.shape[1] > config.max_features:
            rf_sel = RandomForestClassifier(n_estimators=config.n_estimators,
                                            random_state=42, max_depth=10)
            rf_sel.fit(X_train_scaled, y_train)
            top_indices = np.argsort(rf_sel.feature_importances_)[::-1][:config.max_features]
            X_train_sel = X_train[:, top_indices]
            X_val_sel   = X_val[:,   top_indices]
        else:
            X_train_sel = X_train
            X_val_sel   = X_val

        scaler_after       = StandardScaler()
        X_train_sel_scaled = np.nan_to_num(scaler_after.fit_transform(X_train_sel),
                                           nan=0., posinf=0., neginf=0.)
        X_val_sel_scaled   = np.nan_to_num(scaler_after.transform(X_val_sel),
                                           nan=0., posinf=0., neginf=0.)

        rf_after = RandomForestClassifier(n_estimators=config.n_estimators,
                                          random_state=42, max_depth=10)
        rf_after.fit(X_train_sel_scaled, y_train)
        pred_after = rf_after.predict(X_val_sel_scaled)
        acc_scores_after.append(accuracy_score(y_val, pred_after))
        f1_scores_after.append(f1_score(y_val, pred_after, average='weighted'))

    return (np.mean(acc_scores_before), np.std(acc_scores_before),
            np.mean(f1_scores_before),  np.std(f1_scores_before),
            np.mean(acc_scores_after),  np.std(acc_scores_after),
            np.mean(f1_scores_after),   np.std(f1_scores_after))


def run_optimized_experiment(dataset_name: str, config: Config) -> Dict:
    print(f"\n=== {dataset_name} ===")
    X, y = load_classification(dataset_name)
    print(f"  {len(X)} samples  {X[0].shape[0]} raw ch  T={X[0].shape[1]}  "
          f"{len(np.unique(y))} classes")

    # augment once, reuse for both signature and area/vol features
    X_aug = [augment_channels(x) for x in X]

    base_features = np.nan_to_num(
        np.array([iisignature.sig(x.T, config.sig_level) for x in X_aug]),
        nan=0., posinf=1e6, neginf=-1e6)

    signed_features, unsigned_features = batch_extract_features(X, config)

    feature_sets = {
        'base':                 base_features,
        'signed':               signed_features,
        'unsigned':             unsigned_features,
        'base_signed':          np.concatenate([base_features, signed_features],   axis=1),
        'base_unsigned':        np.concatenate([base_features, unsigned_features], axis=1),
        'signed_unsigned':      np.concatenate([signed_features, unsigned_features], axis=1),
        'base_signed_unsigned': np.concatenate([base_features, signed_features,
                                                unsigned_features], axis=1),
    }

    results = {}
    for name, features in feature_sets.items():
        (ab, sb, fb, sfb,
         aa, sa, fa, sfa) = evaluate_with_cv(features, y, config)
        results[name] = {
            'before_selection': {
                'acc_mean': ab, 'acc_std': sb,
                'f1_mean':  fb, 'f1_std':  sfb,
                'n_features': features.shape[1],
            },
            'after_selection': {
                'acc_mean': aa, 'acc_std': sa,
                'f1_mean':  fa, 'f1_std':  sfa,
                'n_features': min(features.shape[1], config.max_features),
            },
        }

    print(f"\n{'Feature Set':<20} {'Before Selection':<25} {'After Selection':<25} {'Features':<15}")
    print(f"{'':20} {'Accuracy':>12} {'F1-Score':>12} {'Accuracy':>12} {'F1-Score':>12} {'(orig→sel)':<15}")
    print("-" * 95)
    for name in ['base', 'signed', 'unsigned', 'signed_unsigned',
                 'base_signed', 'base_unsigned', 'base_signed_unsigned']:
        if name not in results: continue
        before = results[name]['before_selection']
        after  = results[name]['after_selection']
        print(f"{name:<20} "
              f"{before['acc_mean']:.4f}±{before['acc_std']:.4f} "
              f"{before['f1_mean']:.4f}±{before['f1_std']:.4f} "
              f"{after['acc_mean']:.4f}±{after['acc_std']:.4f} "
              f"{after['f1_mean']:.4f}±{after['f1_std']:.4f} "
              f"{before['n_features']}→{after['n_features']}")
    return results


# ════════════════════════════════════════════════════════════════════════════
# Multi-dataset runner
# ════════════════════════════════════════════════════════════════════════════

DATASETS = [
    "AtrialFibrillation", "HandMovementDirection", "Handwriting", "LSST",
    "Libras", "SelfRegulationSCP1", "SelfRegulationSCP2", "StandWalkJump",
    "ArticularyWordRecognition", "BasicMotions", "ERing", "NATOPS",
    "PenDigits", "RacketSports", "UWaveGestureLibrary",
]

FEATURE_SETS = [
    "base", "signed", "unsigned", "signed_unsigned",
    "base_signed", "base_unsigned", "base_signed_unsigned",
]

config = Config(
    sig_level=2,
    start_k=2,
    max_k=2,
    max_combinations_per_k=10000000000000000000,
    max_features=250,
)

all_results = {}
for ds in DATASETS:
    print(f"\n{'='*80}\nDATASET: {ds}\n{'='*80}")
    try:
        all_results[ds] = run_optimized_experiment(ds, config)
    except Exception as exc:
        print(f"   ERROR: {exc}")
        all_results[ds] = None


def cell_val(ds_results, fs, phase, metric):
    if ds_results is None or fs not in ds_results: return "NaN"
    m = ds_results[fs][phase]
    return f"{m[metric+'_mean']:.4f}±{m[metric+'_std']:.4f}"

def feat_count(ds_results, fs, phase):
    if ds_results is None or fs not in ds_results: return 0
    return ds_results[fs][phase]["n_features"]

phases   = ["before_selection", "after_selection"]
p_labels = {"before_selection": "before", "after_selection": "after"}

acc_rows, f1_rows, feat_rows = [], [], []
for ds in DATASETS:
    dr      = all_results.get(ds)
    acc_row = {"dataset": ds}; f1_row = {"dataset": ds}; feat_row = {"dataset": ds}
    for fs in FEATURE_SETS:
        for ph in phases:
            col           = f"{fs}_{p_labels[ph]}"
            acc_row[col]  = cell_val(dr, fs, ph, "acc")
            f1_row[col]   = cell_val(dr, fs, ph, "f1")
            feat_row[col] = feat_count(dr, fs, ph)
    acc_rows.append(acc_row); f1_rows.append(f1_row); feat_rows.append(feat_row)

acc_df  = pd.DataFrame(acc_rows).set_index("dataset")
f1_df   = pd.DataFrame(f1_rows).set_index("dataset")
feat_df = pd.DataFrame(feat_rows).set_index("dataset")

def print_table(df, title):
    print(f"\n{'='*120}\n  {title}\n{'='*120}")
    print(df.to_string())
    print()

print_table(acc_df,  "ACCURACY TABLE  (mean±std, 5-fold CV, aug=8x(x,chord,curv,|x|)+norm, sig_level=2, max_k=2)")
print_table(f1_df,   "F1-SCORE TABLE  (mean±std, 5-fold CV, aug=8x(x,chord,curv,|x|)+norm, sig_level=2, max_k=2)")
print_table(feat_df, "FEATURE COUNT TABLE  (before → after RF-importance selection)")

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
acc_df.to_csv(f"accuracy_table_{ts}.csv")
f1_df.to_csv(f"f1_table_{ts}.csv")
feat_df.to_csv(f"feature_counts_table_{ts}.csv")
print(f"Saved: accuracy_table_{ts}.csv")
print(f"Saved: f1_table_{ts}.csv")
print(f"Saved: feature_counts_table_{ts}.csv")
